# BiaPy as ops: a model, then a watershed

Two biapy-ops in a row, run by scikit-ops' runner in the `biapy` environment:

1. **`predict.bmz`** runs a BioImage Model Zoo model -- here `resourceful-otter`,
   2D nuclei -- and returns the network's raw channels: F (foreground) and C (contours).
2. **`segment.watershed_by_channels`** turns those channels into labels.

Because the second step is its own op, the thresholds can be tuned without
running the network again. The first run builds the environment (PyTorch and
BiaPy, several GB) and downloads the model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from skimage.data import human_mitosis

import oprunner
from biapy_ops.predict import Output, bmz
from biapy_ops.segment import watershed_by_channels

image = human_mitosis()[..., np.newaxis].astype(np.float32)  # (y, x, c)
image.shape

## Where each op runs

Asking builds nothing.

In [ ]:
with oprunner.Runner() as runner:
    for op in (bmz, watershed_by_channels):
        env = runner.env_for(op)
        print(f"{op.__module__}:{op.__name__}")
        print(f"    environment: {env.id}")
        print(f"    defined in:  {env.config}")

## The pipeline

One runner for both, so the second op reuses the warm worker the first started.

In [ ]:
with oprunner.Runner() as runner:
    channels = runner.run(bmz, image=image, model="resourceful-otter", output=Output.raw)
    auto = runner.run(watershed_by_channels, prediction=channels, channels="F,C")
    strict = runner.run(
        watershed_by_channels,
        prediction=channels,
        channels="F,C",
        seed_thresholds="0.8,0.2",  # seeds: very sure foreground, far from any contour
        min_seed_size=20,
    )
channels.shape, auto.max(), strict.max()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
panels = [
    (image[..., 0], "image", "gray"),
    (channels[..., 0], "F (foreground)", "magma"),
    (channels[..., 1], "C (contours)", "magma"),
    (auto, f"auto thresholds: {auto.max()} objects", "nipy_spectral"),
    (strict, f"0.8, 0.2: {strict.max()} objects", "nipy_spectral"),
]
for ax, (data, title, cmap) in zip(axes, panels):
    ax.imshow(data, cmap=cmap, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## The same, in one call

`output=final` (the default) lets BiaPy do the watershed itself, with the
thresholds the model's RDF carries.

In [ ]:
with oprunner.Runner() as runner:
    final = runner.run(bmz, image=image, model="resourceful-otter")
final.shape, final.max()